# 🚀  ADK  🚀


- **Build a Foundational Agent**: Create a simple but effective AI agent from scratch using the Google Agent Development Kit (ADK).

- **Grant New Skills with Custom Tools**: Teach an agent to perform new tasks by connecting it to external APIs, like a real-time weather service.

- **Create a Team of Agents**: Assemble a multi-agent system where a primary agent can delegate specialized tasks to other agents.

- **Master Conversational Memory**: Understand the critical role of Sessions in enabling agents to remember previous interactions, handle feedback, and carry on a coherent conversation.



## Part 0: Setup & Authentication 🔑

First things first, let's get all our tools ready. This step installs the necessary libraries and securely configures your Google API key so your agents can access the power of Gemini.

In [1]:
!pip install google-adk google-generativeai -q

# --- Import all necessary libraries for our entire adventure ---
import os
import re
import asyncio
from IPython.display import display, Markdown
import google.generativeai as genai
from google.adk.agents import Agent
from google.adk.tools import google_search
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService, Session
from google.genai.types import Content, Part
from getpass import getpass

print("✅ All libraries are ready to go!")

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


✅ All libraries are ready to go!


In [2]:
# --- Securely Configure Your API Key ---

# Prompt the user for their API key securely
api_key = getpass('Enter your Google API Key: ')

# Configure the generative AI library with the provided key
genai.configure(api_key=api_key)

# Set the API key as an environment variable for ADK to use
os.environ['GOOGLE_API_KEY'] = api_key

print("✅ API Key configured successfully! Let the fun begin.")

Enter your Google API Key: ··········
✅ API Key configured successfully! Let the fun begin.


---
## Part 1: Your First Agent - The Day Trip Genie 🧞

Meet your first creation! The `day_trip_agent` is a simple but powerful assistant. We're making it a little smarter by teaching it to understand **budget constraints**.

* **Agent**: The brain of the operation, defined by its instructions, tools, and the AI model it uses.
* **Session**: The conversation history. For this simple agent, it's just a container for a single request-response.
* **Runner**: The engine that connects the `Agent` and the `Session` to process your request and get a response.

```
+--------------------------------------------------+
|         Spontaneous Day Trip Agent 🤖            |
|--------------------------------------------------|
|  Model: gemini-2.5-flash                         |
|  Description:                                    |
|   Generates full-day trip itineraries based on   |
|   mood, interests, and budget                    |
|--------------------------------------------------|
|  🔧 Tools:                                       |
|   - Google Search                                |
|--------------------------------------------------|
|  🧠 Capabilities:                                |
|   - Budget Awareness (cheap / splurge)           |
|   - Mood Matching (adventurous, relaxing, etc.)  |
|   - Real-Time Info (hours, events)               |
|   - Morning / Afternoon / Evening plan           |
+--------------------------------------------------+

            ▲
            |
    +------------------+
    |   User Input     |
    |------------------|
    |  Mood            |
    |  Interests       |
    |  Budget          |
    +------------------+

            |
            ▼

+--------------------------------------------------+
|             Output: Markdown Itinerary           |
|--------------------------------------------------|
| - Time blocks (Morning / Afternoon / Evening)    |
| - Venue names with links and hours               |
| - Budget-matching activities                     |
+--------------------------------------------------+
```


In [3]:
# --- Agent Definition ---

def create_day_trip_agent():
    """Create the Spontaneous Day Trip Generator agent"""
    return Agent(
        name="day_trip_agent",
        model="gemini-2.5-flash",
        description="Agent specialized in generating spontaneous full-day itineraries based on mood, interests, and budget.",
        instruction="""
        You are the "Spontaneous Day Trip" Generator 🚗 - a specialized AI assistant that creates engaging full-day itineraries.

        Your Mission:
        Transform a simple mood or interest into a complete day-trip adventure with real-time details, while respecting a budget.

        Guidelines:
        1. **Budget-Aware**: Pay close attention to budget hints like 'cheap', 'affordable', or 'splurge'. Use Google Search to find activities (free museums, parks, paid attractions) that match the user's budget.
        2. **Full-Day Structure**: Create morning, afternoon, and evening activities.
        3. **Real-Time Focus**: Search for current operating hours and special events.
        4. **Mood Matching**: Align suggestions with the requested mood (adventurous, relaxing, artsy, etc.).

        RETURN itinerary in MARKDOWN FORMAT with clear time blocks and specific venue names.
        """,
        tools=[google_search]
    )

day_trip_agent = create_day_trip_agent()
print(f"🧞 Agent '{day_trip_agent.name}' is created and ready for adventure!")

🧞 Agent 'day_trip_agent' is created and ready for adventure!


In [4]:
# --- A Helper Function to Run Our Agents ---
# We'll use this function throughout the notebook to make running queries easy.

async def run_agent_query(agent: Agent, query: str, session: Session, user_id: str, is_router: bool = False):
    """Initializes a runner and executes a query for a given agent and session."""
    print(f"\n🚀 Running query for agent: '{agent.name}' in session: '{session.id}'...")

    runner = Runner(
        agent=agent,
        session_service=session_service,
        app_name=agent.name
    )

    final_response = ""
    try:
        async for event in runner.run_async(
            user_id=user_id,
            session_id=session.id,
            new_message=Content(parts=[Part(text=query)], role="user")
        ):
            if not is_router:
                # Let's see what the agent is thinking!
                print(f"EVENT: {event}")
            if event.is_final_response():
                final_response = event.content.parts[0].text
    except Exception as e:
        final_response = f"An error occurred: {e}"

    if not is_router:
     print("\n" + "-"*50)
     print("✅ Final Response:")
     display(Markdown(final_response))
     print("-"*50 + "\n")

    return final_response

# --- Initialize our Session Service ---
# This one service will manage all the different sessions in our notebook.
session_service = InMemorySessionService()
my_user_id = "adk_adventurer_001"

In [5]:
# --- Let's test the Day Trip Genie! ---

async def run_day_trip_genie():
    # Create a new, single-use session for this query
    day_trip_session = await session_service.create_session(
        app_name=day_trip_agent.name,
        user_id=my_user_id
    )

    # Note the new budget constraint in the query!
    query = "Plan a relaxing and artsy day trip near Sunnyvale, CA. Keep it affordable!"
    print(f"🗣️ User Query: '{query}'")

    await run_agent_query(day_trip_agent, query, day_trip_session, my_user_id)

await run_day_trip_genie()

🗣️ User Query: 'Plan a relaxing and artsy day trip near Sunnyvale, CA. Keep it affordable!'

🚀 Running query for agent: 'day_trip_agent' in session: '0d5cc04e-fa68-4ec7-b530-fca2e96c410a'...
EVENT: model_version='gemini-2.5-flash' content=Content(
  parts=[
    Part(
      text="""Here's a relaxing and artsy, yet affordable, day trip itinerary near Sunnyvale, CA:

## Relaxing & Artsy Day Trip near Sunnyvale, CA

This itinerary focuses on free and low-cost artistic experiences combined with opportunities for relaxation and affordable dining, making the most of a Monday visit.

### Morning (10:30 AM - 1:00 PM): Artistic Immersion at Stanford University

*   **10:30 AM - 1:00 PM: Cantor Arts Center & Rodin Sculpture Garden (Stanford, CA)**
    Start your day with a visit to the **Cantor Arts Center** at Stanford University. Admission is always free, offering access to a vast collection spanning 5,000 years of art. Don't miss the outdoor **Rodin Sculpture Garden**, which houses one of the 

Here's a relaxing and artsy, yet affordable, day trip itinerary near Sunnyvale, CA:

## Relaxing & Artsy Day Trip near Sunnyvale, CA

This itinerary focuses on free and low-cost artistic experiences combined with opportunities for relaxation and affordable dining, making the most of a Monday visit.

### Morning (10:30 AM - 1:00 PM): Artistic Immersion at Stanford University

*   **10:30 AM - 1:00 PM: Cantor Arts Center & Rodin Sculpture Garden (Stanford, CA)**
    Start your day with a visit to the **Cantor Arts Center** at Stanford University. Admission is always free, offering access to a vast collection spanning 5,000 years of art. Don't miss the outdoor **Rodin Sculpture Garden**, which houses one of the largest collections of Rodin bronze sculptures outside of Paris, including "The Thinker". Strolling through the outdoor garden provides a wonderfully relaxing and inspiring experience. As today is Monday, the museum is open from 11:00 a.m. to 6:00 p.m..

    *   **Cost:** Free admission. Parking at Stanford is paid on weekdays via the ParkMobile app (enforced Monday-Friday, 8 AM - 4 PM).
    *   **Mood:** Artsy, Relaxing, Inspiring.

### Lunch (1:00 PM - 2:30 PM): Affordable Bites in Palo Alto

*   **1:00 PM - 2:30 PM: Lunch on California Avenue or University Avenue (Palo Alto, CA)**
    Head to nearby Palo Alto for an affordable and delicious lunch. Both California Avenue and University Avenue offer a variety of budget-friendly eateries.
    *   **Recommendation:** Consider **Oren's Hummus** for fresh and generous portions of Mediterranean cuisine, or **Mediterranean Wraps** for tasty and filling options. **Curry Up Now** is also a good choice for modern Indian street food. Most meals at these spots are wallet-friendly.
    *   **Cost:** ~$10-$20 per person.
    *   **Mood:** Relaxed, Local Flavor.

### Afternoon (2:30 PM - 5:30 PM): Public Art & Nature Walk

*   **2:30 PM - 4:30 PM: Stanford University Public Art & Campus Walk (Stanford, CA)**
    After lunch, return to the Stanford campus for a relaxing walk to discover its extensive outdoor public art collection. The university campus features over 80 works of outdoor public art, accessible daily. You can explore using the **StanfordMobile app** for a self-guided tour and map highlighting the artworks. This allows for a leisurely pace, combining artistic appreciation with a serene stroll through the beautiful campus grounds.
    *   **Cost:** Free.
    *   **Mood:** Artsy, Relaxing, Serene.

*   **4:30 PM - 5:30 PM: Sunnyvale Baylands Park (Sunnyvale, CA)**
    For a truly relaxing end to your afternoon, drive back to Sunnyvale and visit **Sunnyvale Baylands Park**. This expansive park, situated along the shores of San Francisco Bay, offers tranquil scenic walks, birdwatching opportunities, and diverse ecosystems. It's a perfect spot to unwind, breathe in fresh air, and enjoy the natural beauty of the South Bay before dinner.
    *   **Cost:** Free.
    *   **Mood:** Relaxing, Nature-focused, Peaceful.

### Evening (6:00 PM Onwards): Affordable Dinner & Leisurely Evening

*   **6:00 PM - 7:30 PM: Dinner on Castro Street (Mountain View, CA)**
    Head to Mountain View's vibrant Castro Street for a wide selection of affordable dinner options. Mountain View is known for its budget-friendly restaurants.
    *   **Recommendation:** Try **Taqueria La Espuela** for authentic and generously portioned Mexican food, with most meals under $10. **The Falafel Stop** in Sunnyvale (close to Mountain View) also offers quick, delicious, and affordable Mediterranean wraps. For a different flavor, **Chef Zhao Kitchen** in Mountain View provides quality Chinese cuisine at reasonable prices, generally $8-$12 per dish.
    *   **Cost:** ~$10-$20 per person.
    *   **Mood:** Casual, Satisfying.

*   **7:30 PM Onwards: Evening Stroll or Relax at a Local Cafe**
    After dinner, enjoy a leisurely stroll along Castro Street, which often has a pleasant evening ambiance. Alternatively, find a cozy, affordable cafe for a warm drink and some quiet reflection to end your relaxing and artsy day trip.
    *   **Cost:** Varies based on purchases.
    *   **Mood:** Relaxing, Reflective.

--------------------------------------------------



---
## Part 2: Supercharging Agents with Custom Tools 🛠️

So far, we've used the powerful built-in `GoogleSearch` tool. But the true power of agents comes from connecting them to your own logic and data sources.

This is where **custom tools** come in. Let's explore three patterns for giving your agent new skills, using real-world, practical examples.

### 2.1 The Simple `FunctionTool`: Calling a Real-Time Weather API

The most direct way to create a tool is by writing a Python function. This is perfect for synchronous tasks like fetching data from an API.

**Key Concept:** The function's **docstring** is critical. The ADK uses it as the tool's official description, which the LLM reads to understand its purpose, parameters, and when to use it.

In this example, we'll create a tool that calls the **free, public U.S. National Weather Service API** to get a real-time forecast. No API key needed!

In [6]:
# --- Tool Definition: A function that calls a live public API ---
import requests
import json

# A simple lookup to avoid needing a separate geocoding API for this example
LOCATION_COORDINATES = {
    "sunnyvale": "37.3688,-122.0363",
    "san francisco": "37.7749,-122.4194",
    "lake tahoe": "39.0968,-120.0324"
}

def get_live_weather_forecast(location: str) -> dict:
    """Gets the current, real-time weather forecast for a specified location in the US.

    Args:
        location: The city name, e.g., "San Francisco".

    Returns:
        A dictionary containing the temperature and a detailed forecast.
    """
    print(f"🛠️ TOOL CALLED: get_live_weather_forecast(location='{location}')")

    # Find coordinates for the location
    normalized_location = location.lower()
    coords_str = None
    for key, val in LOCATION_COORDINATES.items():
        if key in normalized_location:
            coords_str = val
            break
    if not coords_str:
        return {"status": "error", "message": f"I don't have coordinates for {location}."}

    try:
        # NWS API requires 2 steps: 1. Get the forecast URL from the coordinates.
        points_url = f"https://api.weather.gov/points/{coords_str}"
        headers = {"User-Agent": "ADK Example Notebook"}
        points_response = requests.get(points_url, headers=headers)
        points_response.raise_for_status() # Raise an exception for bad status codes
        forecast_url = points_response.json()['properties']['forecast']

        # 2. Get the actual forecast from the URL.
        forecast_response = requests.get(forecast_url, headers=headers)
        forecast_response.raise_for_status()

        # Extract the relevant forecast details
        current_period = forecast_response.json()['properties']['periods'][0]
        return {
            "status": "success",
            "temperature": f"{current_period['temperature']}°{current_period['temperatureUnit']}",
            "forecast": current_period['detailedForecast']
        }
    except requests.exceptions.RequestException as e:
        return {"status": "error", "message": f"API request failed: {e}"}

# --- Agent Definition: An agent that USES the new tool ---

weather_agent = Agent(
    name="weather_aware_planner",
    model="gemini-2.5-flash",
    description="A trip planner that checks the real-time weather before making suggestions.",
    instruction="You are a cautious trip planner. Before suggesting any outdoor activities, you MUST use the `get_live_weather_forecast` tool to check conditions. Incorporate the live weather details into your recommendation.",
    tools=[get_live_weather_forecast]
)

print(f"🌦️ Agent '{weather_agent.name}' is created and can now call a live weather API!")

🌦️ Agent 'weather_aware_planner' is created and can now call a live weather API!


In [7]:
# --- Let's test the Weather-Aware Planner ---

async def run_weather_planner_test():
    weather_session = await session_service.create_session(app_name=weather_agent.name, user_id=my_user_id)
    query = "I want to go hiking near Lake Tahoe, what's the weather like?"
    print(f"🗣️ User Query: '{query}'")
    await run_agent_query(weather_agent, query, weather_session, my_user_id)

await run_weather_planner_test()

🗣️ User Query: 'I want to go hiking near Lake Tahoe, what's the weather like?'

🚀 Running query for agent: 'weather_aware_planner' in session: '2fdf4f7c-3977-4211-a888-b462a89e0b55'...


EVENT: model_version='gemini-2.5-flash' content=Content(
  parts=[
    Part(
      function_call=FunctionCall(
        args={
          'location': 'Lake Tahoe'
        },
        id='adk-ce7b110f-9b01-4206-a9f7-f3f6c5051090',
        name='get_live_weather_forecast'
      ),
      thought_signature=b"\n\xbe\x02\x01\xbe>\xf6\xfb\xe7\xea,\x9f\xa6\xc5F\x81\xd5c\xe5\xc8\xfd\xcaL\xf9lV\xc7&Y@\\\xdb'\xa5\x07\xf1\xdc\x18\xf9+vnM\x1d[\xc0\x1e\xf8I\x16\x05\xa9\xb3\x94\x9c\x12\xf9\xe0\xd5Ul\xba\x00\x17\x01\x89R\x99e\x00x$R\xa1V\x01E\x9e\x03I\xaa\xf7\xd7\x9f@\x16\xa8\xce\x04\xc9\xc702tA\xb6\x8e...'
    ),
  ],
  role='model'
) grounding_metadata=None partial=None turn_complete=None finish_reason=<FinishReason.STOP: 'STOP'> error_code=None error_message=None interrupted=None custom_metadata=None usage_metadata=GenerateContentResponseUsageMetadata(
  candidates_token_count=20,
  prompt_token_count=187,
  prompt_tokens_details=[
    ModalityTokenCount(
      modality=<MediaModality.TEXT: 'TEXT'>,
 

The weather near Lake Tahoe is sunny, with a high near 60°F and a north wind around 0 mph. It sounds like perfect weather for a hike!

--------------------------------------------------



## 2.2 The Agent-as-a-Tool: Consulting a Specialist 🧑‍🍳

Why build one agent that does everything when you can build a **team of specialist agents?** The **Agent-as-a-Tool** pattern allows one agent to delegate a task to another agent.

**Key Concept:** This is different from a sub-agent. When Agent A calls Agent B as a tool, Agent B's response is passed **back to Agent A**. Agent A then uses that information to form its own final response to the user. It's a powerful way to compose complex behaviors from simpler, focused, and reusable agents.

### How It Works

Our top-level agent, the `trip_data_concierge_agent`, acts as the **Orchestrator**. It has two tools at its disposal:

1.  `call_db_agent`: A function that internally calls our `db_agent` to fetch raw data.
2.  `call_concierge_agent`: A function that calls the `concierge_agent`.

The `concierge_agent` itself has a tool: the `food_critic_agent`.

The flow for a complex query is:

1.  **User** asks the `trip_data_concierge_agent` for a hotel and a nearby restaurant.
2.  The **Orchestrator** first calls `call_db_agent` to get hotel data.
3.  The data is saved in `tool_context.state`.
4.  The **Orchestrator** then calls `call_concierge_agent`, which retrieves the hotel data from the context.
5.  The `concierge_agent` receives the request and decides it needs to use its own tool, the `food_critic_agent`.
6.  The `food_critic_agent` provides a witty recommendation.
7.  The `concierge_agent` gets the critic's response and politely formats it.
8.  This final, polished response is returned to the **Orchestrator**, which presents it to the user.

                         +-----------------------------------------------------------+
                         |              🧭 Trip Data Concierge Agent                 |
                         |-----------------------------------------------------------|
                         |  Model: gemini-2.5-flash                                  |
                         |  Description:                                             |
                         |   Orchestrates database query and travel recommendation  |
                         |-----------------------------------------------------------|
                         |  🔧 Tools:                                                |
                         |   1. call_db_agent                                        |
                         |   2. call_concierge_agent                                 |
                         +-----------------------------------------------------------+
                                      /                                \
                                     /                                  \
                                    ▼                                    ▼
        +-------------------------------------------+    +---------------------------------------------+
        |            🔧 Tool: call_db_agent         |    |         🔧 Tool: call_concierge_agent        |
        |-------------------------------------------|    |---------------------------------------------|
        | Calls: db_agent                           |    | Calls: concierge_agent                       |
        |                                           |    | Uses data from db_agent for recommendations |
        +-------------------------------------------+    +---------------------------------------------+
                                |                                          |
                                ▼                                          ▼
       +--------------------------------------------+   +------------------------------------------------+
       |              📦 db_agent                   |   |             🤵 concierge_agent                  |
       |--------------------------------------------|   |------------------------------------------------|
       | Model: gemini-2.5-flash                    |   | Model: gemini-2.5-flash                         |
       | Role: Return mock JSON hotel data          |   | Role: Hotel staff that handles user Q&A        |
       +--------------------------------------------+   | Tools:                                          |
                                                         |  - food_critic_agent                           |
                                                         +------------------------------------------------+
                                                                                 |
                                                                                 ▼
                                                       +------------------------------------------------+
                                                       |          🍽️ food_critic_agent                  |
                                                       |------------------------------------------------|
                                                       | Model: gemini-2.5-flash                         |
                                                       | Role: Gives a witty restaurant recommendation   |
                                                       +------------------------------------------------+


In [8]:
import asyncio
from google.adk.tools import ToolContext
from google.adk.tools.agent_tool import AgentTool

# Assume 'db_agent' is a pre-defined NL2SQL Agent
# For this example, we'll create placeholder agents.

db_agent = Agent(
    name="db_agent",
    model="gemini-2.5-flash",
    instruction="You are a database agent. When asked for data, return this mock JSON object: {'status': 'success', 'data': [{'name': 'The Grand Hotel', 'rating': 5, 'reviews': 450}, {'name': 'Seaside Inn', 'rating': 4, 'reviews': 620}]}")

# --- 1. Define the Specialist Agents ---

# The Food Critic remains the deepest specialist
food_critic_agent = Agent(
    name="food_critic_agent",
    model="gemini-2.5-flash",
    instruction="You are a snobby but brilliant food critic. You ONLY respond with a single, witty restaurant suggestion near the provided location.",
)

# The Concierge knows how to use the Food Critic
concierge_agent = Agent(
    name="concierge_agent",
    model="gemini-2.5-flash",
    instruction="You are a five-star hotel concierge. If the user asks for a restaurant recommendation, you MUST use the `food_critic_agent` tool. Present the opinion to the user politely.",
    tools=[AgentTool(agent=food_critic_agent)]
)


# --- 2. Define the Tools for the Orchestrator ---

async def call_db_agent(
    question: str,
    tool_context: ToolContext,
):
    """
    Use this tool FIRST to connect to the database and retrieve a list of places, like hotels or landmarks.
    """
    print("--- TOOL CALL: call_db_agent ---")
    agent_tool = AgentTool(agent=db_agent)
    db_agent_output = await agent_tool.run_async(
        args={"request": question}, tool_context=tool_context
    )
    # Store the retrieved data in the context's state
    tool_context.state["retrieved_data"] = db_agent_output
    return db_agent_output


async def call_concierge_agent(
    question: str,
    tool_context: ToolContext,
):
    """
    After getting data with call_db_agent, use this tool to get travel advice, opinions, or recommendations.
    """
    print("--- TOOL CALL: call_concierge_agent ---")
    # Retrieve the data fetched by the previous tool
    input_data = tool_context.state.get("retrieved_data", "No data found.")

    # Formulate a new prompt for the concierge, giving it the data context
    question_with_data = f"""
    Context: The database returned the following data: {input_data}

    User's Request: {question}
    """

    agent_tool = AgentTool(agent=concierge_agent)
    concierge_output = await agent_tool.run_async(
        args={"request": question_with_data}, tool_context=tool_context
    )
    return concierge_output


# --- 3. Define the Top-Level Orchestrator Agent ---

trip_data_concierge_agent = Agent(
    name="trip_data_concierge",
    model="gemini-2.5-flash",
    description="Top-level agent that queries a database for travel data, then calls a concierge agent for recommendations.",
    tools=[call_db_agent, call_concierge_agent],
    instruction="""
    You are a master travel planner who uses data to make recommendations.

    1.  **ALWAYS start with the `call_db_agent` tool** to fetch a list of places (like hotels) that match the user's criteria.

    2.  After you have the data, **use the `call_concierge_agent` tool** to answer any follow-up questions for recommendations, opinions, or advice related to the data you just found.
    """,
)

print(f"✅ Orchestrator Agent '{trip_data_concierge_agent.name}' is defined and ready.")

✅ Orchestrator Agent 'trip_data_concierge' is defined and ready.


In [9]:
# --- Let's test the Trip Data Concierge Agent ---

async def run_trip_data_concierge():
    """
    Sets up a session and runs a query against the top-level
    trip_data_concierge_agent.
    """
    # Create a new, single-use session for this query
    concierge_session = await session_service.create_session(
        app_name=trip_data_concierge_agent.name,
        user_id=my_user_id
    )

    # This query is specifically designed to trigger the full two-step process:
    # 1. Get data from the db_agent.
    # 2. Get a recommendation from the concierge_agent based on that data.
    query = "Find the top-rated hotels in San Francisco from the database, then suggest a dinner spot near the one with the most reviews."
    print(f"🗣️ User Query: '{query}'")

    # We call our existing helper function with the top-level orchestrator agent
    await run_agent_query(trip_data_concierge_agent, query, concierge_session, my_user_id)

# Run the test
await run_trip_data_concierge()

🗣️ User Query: 'Find the top-rated hotels in San Francisco from the database, then suggest a dinner spot near the one with the most reviews.'

🚀 Running query for agent: 'trip_data_concierge' in session: 'a00af0d3-b2ef-4a44-b6b2-d0175da1ded9'...
EVENT: model_version='gemini-2.5-flash' content=Content(
  parts=[
    Part(
      function_call=FunctionCall(
        args={
          'question': 'top-rated hotels in San Francisco'
        },
        id='adk-b931f3b8-513a-4aad-8d0d-c0f64cca8eef',
        name='call_db_agent'
      ),
      thought_signature=b"\n\x96\x06\x01\xbe>\xf6\xfb\xc8G\x0f8\x1d\xea\xfd\xe1i\x9fi\x98\xb1\x07\x07\xeb\x92\xe8\xcc\xa6\xcc\xc0\x1dG#J~q\x1c\x86*\n+\xb1~\x90\xf5\xa8#9\x7fE\x12\xc4\xc6G\xd75\xde\x1c\xa0\x02\xb10\x95\x1e\xe48\x02_\xde\xe24\x9d\xe4I\xcb\xab' ]>\xe0\x9c\x8a\xb5\x88\xac\xb2\xbd\x9d\xa7\xb9\x1fc\xae\xdb\xe7)...'
    ),
  ],
  role='model'
) grounding_metadata=None partial=None turn_complete=None finish_reason=<FinishReason.STOP: 'STOP'> error_code=

I couldn't find any hotels in San Francisco with the provided information.

--------------------------------------------------



---
## Part 3: Agent with a Memory - The Adaptive Planner 🗺️

Now, let's see an agent that not only **remembers** but also **adapts**. We'll challenge the `multi_day_trip_agent` to re-plan part of its itinerary based on our feedback. This is a much more realistic test of conversational AI.

```
+-----------------------------------------------------+
|         Adaptive Multi-Day Trip Agent 🗺️           |
|-----------------------------------------------------|
|  Model: gemini-2.5-flash                            |
|  Description:                                       |
|   Builds multi-day travel itineraries step-by-step, |
|   remembers previous days, adapts to feedback       |
|-----------------------------------------------------|
|  🔧 Tools:                                          |
|   - Google Search                                   |
|-----------------------------------------------------|
|  🧠 Capabilities:                                   |
|   - Memory of past conversation & preferences       |
|   - Progressive planning (1 day at a time)          |
|   - Adapts to user feedback                         |
|   - Ensures activity variety across days            |
+-----------------------------------------------------+

            ▲
            |
    +---------------------------+
    |     User Interaction      |
    |---------------------------|
    | - Destination             |
    | - Trip duration           |
    | - Interests & feedback    |
    +---------------------------+

            |
            ▼

+-----------------------------------------------------+
|        Day-by-Day Itinerary Generation              |
|-----------------------------------------------------|
|  🗓️ Day N Output (Markdown format):                 |
|   - Morning / Afternoon / Evening activities        |
|   - Personalized & context-aware                    |
|   - Changes accepted, feedback acknowledged         |
+-----------------------------------------------------+

            |
            ▼

+-----------------------------------------------------+
|        Next Day Planning Triggered 🚀               |
|-----------------------------------------------------|
| - Builds on prior days                              |
| - Avoids repetition                                 |
| - Asks user for confirmation before proceeding      |
+-----------------------------------------------------+
```


In [10]:
# --- Agent Definition: The Adaptive Planner ---

def create_multi_day_trip_agent():
    """Create the Progressive Multi-Day Trip Planner agent"""
    return Agent(
        name="multi_day_trip_agent",
        model="gemini-2.5-flash",
        description="Agent that progressively plans a multi-day trip, remembering previous days and adapting to user feedback.",
        instruction="""
        You are the "Adaptive Trip Planner" 🗺️ - an AI assistant that builds multi-day travel itineraries step-by-step.

        Your Defining Feature:
        You have short-term memory. You MUST refer back to our conversation to understand the trip's context, what has already been planned, and the user's preferences. If the user asks for a change, you must adapt the plan while keeping the unchanged parts consistent.

        Your Mission:
        1.  **Initiate**: Start by asking for the destination, trip duration, and interests.
        2.  **Plan Progressively**: Plan ONLY ONE DAY at a time. After presenting a plan, ask for confirmation.
        3.  **Handle Feedback**: If a user dislikes a suggestion (e.g., "I don't like museums"), acknowledge their feedback, and provide a *new, alternative* suggestion for that time slot that still fits the overall theme.
        4.  **Maintain Context**: For each new day, ensure the activities are unique and build logically on the previous days. Do not suggest the same things repeatedly.
        5.  **Final Output**: Return each day's itinerary in MARKDOWN format.
        """,
        tools=[google_search]
    )

multi_day_agent = create_multi_day_trip_agent()
print(f"🗺️ Agent '{multi_day_agent.name}' is created and ready to plan and adapt!")

🗺️ Agent 'multi_day_trip_agent' is created and ready to plan and adapt!


### Scenario 3a: Agent WITH Memory (Using a SINGLE Session) ✅

First, let's see the correct way to do it. We will use the **exact same `trip_session` object** for the entire conversation. Watch how the agent remembers the context from Turn 1 to correctly handle the requests in Turn 2 and 3.

In [11]:
# --- Scenario 2: Testing Adaptation and Memory ---

async def run_adaptive_memory_demonstration():
    print("### 🧠 DEMO 2: AGENT THAT ADAPTS (SAME SESSION) ###")

    # Create ONE session that we will reuse for the whole conversation
    trip_session = await session_service.create_session(
        app_name=multi_day_agent.name,
        user_id=my_user_id
    )
    print(f"Created a single session for our trip: {trip_session.id}")

    # --- Turn 1: The user initiates the trip ---
    query1 = "Hi! I want to plan a 2-day trip to Lisbon, Portugal. I'm interested in historic sites and great local food."
    print(f"\n🗣️ User (Turn 1): '{query1}'")
    await run_agent_query(multi_day_agent, query1, trip_session, my_user_id)

    # --- Turn 2: The user gives FEEDBACK and asks for a CHANGE ---
    # We use the EXACT SAME `trip_session` object!
    query2 = "That sounds pretty good, but I'm not a huge fan of castles. Can you replace the morning activity for Day 1 with something else historical?"
    print(f"\n🗣️ User (Turn 2 - Feedback): '{query2}'")
    await run_agent_query(multi_day_agent, query2, trip_session, my_user_id)

    # --- Turn 3: The user confirms and asks to continue ---
    query3 = "Yes, the new plan for Day 1 is perfect! Please plan Day 2 now, keeping the food theme in mind."
    print(f"\n🗣️ User (Turn 3 - Confirmation): '{query3}'")
    await run_agent_query(multi_day_agent, query3, trip_session, my_user_id)

await run_adaptive_memory_demonstration()

### 🧠 DEMO 2: AGENT THAT ADAPTS (SAME SESSION) ###
Created a single session for our trip: d6c9415c-9215-422d-aad8-e665e08819fe

🗣️ User (Turn 1): 'Hi! I want to plan a 2-day trip to Lisbon, Portugal. I'm interested in historic sites and great local food.'

🚀 Running query for agent: 'multi_day_trip_agent' in session: 'd6c9415c-9215-422d-aad8-e665e08819fe'...
EVENT: model_version='gemini-2.5-flash' content=Content(
  parts=[
    Part(
      text="""Hello! Lisbon is a fantastic choice with its rich history and delicious food. I can definitely help you plan a wonderful 2-day trip.

Let's start with **Day 1**. How about this itinerary focusing on some iconic historic sites and local culinary delights?

**Day 1: Alfama and Baixa's Historic Charm**

*   **Morning (9:00 AM - 1:00 PM): Explore Alfama District & São Jorge Castle**
    *   Begin your day by wandering through the narrow, winding streets of Alfama, Lisbon's oldest district.
    *   Visit the imposing São Jorge Castle for panoramic

Hello! Lisbon is a fantastic choice with its rich history and delicious food. I can definitely help you plan a wonderful 2-day trip.

Let's start with **Day 1**. How about this itinerary focusing on some iconic historic sites and local culinary delights?

**Day 1: Alfama and Baixa's Historic Charm**

*   **Morning (9:00 AM - 1:00 PM): Explore Alfama District & São Jorge Castle**
    *   Begin your day by wandering through the narrow, winding streets of Alfama, Lisbon's oldest district.
    *   Visit the imposing São Jorge Castle for panoramic views of the city and the Tagus River.
*   **Lunch (1:00 PM - 2:30 PM): Traditional Portuguese Lunch in Alfama**
    *   Enjoy a traditional Portuguese lunch at a local tasca (tavern) in Alfama, perhaps trying "Bacalhau à Brás" (codfish with scrambled eggs and potatoes) or fresh grilled sardines.
*   **Afternoon (2:30 PM - 5:30 PM): Lisbon Cathedral & Baixa District**
    *   Descend from Alfama to visit the Lisbon Cathedral (Sé de Lisboa), the city's oldest church.
    *   Stroll through the grid-patterned streets of the Baixa district, rebuilt after the 1755 earthquake, and admire Rossio Square and Praça do Comércio.
*   **Evening (7:00 PM onwards): Dinner & Fado Music Experience**
    *   Savor a delightful dinner in the Baixa or Chiado area.
    *   Consider experiencing a traditional Fado show, a soulful Portuguese musical genre, often performed in intimate restaurants.

How does this sound for your first day in Lisbon?

--------------------------------------------------


🗣️ User (Turn 2 - Feedback): 'That sounds pretty good, but I'm not a huge fan of castles. Can you replace the morning activity for Day 1 with something else historical?'

🚀 Running query for agent: 'multi_day_trip_agent' in session: 'd6c9415c-9215-422d-aad8-e665e08819fe'...
EVENT: model_version='gemini-2.5-flash' content=Content(
  parts=[
    Part(
      text="""Understood! No problem at all. We can certainly swap out the castle for another fantastic historical experience.

How about we replace the São Jorge Castle with a visit to the **Lisbon Cathedral (Sé de Lisboa)** and then explore some of the other historical churches and viewpoints within the Alfama district itself? This way, you still get the historical immersion and great views without the castle.

Here's the revised Day 1 itinerary:

**Day 1: Alfama and Baixa's Historic Charm**

*   **Morning (9:00 AM - 1:00 PM): Explore Alfama District & Lisbon Cathedral**
    *   Begin y

Understood! No problem at all. We can certainly swap out the castle for another fantastic historical experience.

How about we replace the São Jorge Castle with a visit to the **Lisbon Cathedral (Sé de Lisboa)** and then explore some of the other historical churches and viewpoints within the Alfama district itself? This way, you still get the historical immersion and great views without the castle.

Here's the revised Day 1 itinerary:

**Day 1: Alfama and Baixa's Historic Charm**

*   **Morning (9:00 AM - 1:00 PM): Explore Alfama District & Lisbon Cathedral**
    *   Begin your day by wandering through the narrow, winding streets of Alfama, Lisbon's oldest district.
    *   Visit the imposing **Lisbon Cathedral (Sé de Lisboa)**, the city's oldest church, and explore its rich history and architecture.
    *   Continue to explore other historical gems within Alfama, such as the **Miradouro de Santa Luzia** and **Miradouro das Portas do Sol** for stunning views over the terracotta rooftops and the Tagus River, perhaps also passing by the **Church of São Vicente de Fora**.
*   **Lunch (1:00 PM - 2:30 PM): Traditional Portuguese Lunch in Alfama**
    *   Enjoy a traditional Portuguese lunch at a local tasca (tavern) in Alfama, perhaps trying "Bacalhau à Brás" (codfish with scrambled eggs and potatoes) or fresh grilled sardines.
*   **Afternoon (2:30 PM - 5:30 PM): Baixa District & Praça do Comércio**
    *   Descend from Alfama to stroll through the grid-patterned streets of the Baixa district, rebuilt after the 1755 earthquake, and admire **Rossio Square** and **Praça do Comércio**.
*   **Evening (7:00 PM onwards): Dinner & Fado Music Experience**
    *   Savor a delightful dinner in the Baixa or Chiado area.
    *   Consider experiencing a traditional Fado show, a soulful Portuguese musical genre, often performed in intimate restaurants.

How does this updated plan for Day 1 sound to you?

--------------------------------------------------


🗣️ User (Turn 3 - Confirmation): 'Yes, the new plan for Day 1 is perfect! Please plan Day 2 now, keeping the food theme in mind.'

🚀 Running query for agent: 'multi_day_trip_agent' in session: 'd6c9415c-9215-422d-aad8-e665e08819fe'...
EVENT: model_version='gemini-2.5-flash' content=Content(
  parts=[
    Part(
      text="""Excellent! I'm glad Day 1 is perfect. Let's move on to **Day 2**, focusing on more historical gems and, of course, delicious local food experiences.

Here's a plan for your second day in Lisbon:

**Day 2: Belém's Discoveries & Culinary Delights**

*   **Morning (9:00 AM - 1:00 PM): Discover Belém's Maritime History**
    *   Start your day by heading to the historic Belém district, a UNESCO World Heritage site deeply connected to Portugal's Age of Discoveries.
    *   Visit the magnificent **Jerónimos Monastery (Mosteiro dos Jerónimos)**, a stunning example of Manueline architecture and the resting place of Vasco 

Excellent! I'm glad Day 1 is perfect. Let's move on to **Day 2**, focusing on more historical gems and, of course, delicious local food experiences.

Here's a plan for your second day in Lisbon:

**Day 2: Belém's Discoveries & Culinary Delights**

*   **Morning (9:00 AM - 1:00 PM): Discover Belém's Maritime History**
    *   Start your day by heading to the historic Belém district, a UNESCO World Heritage site deeply connected to Portugal's Age of Discoveries.
    *   Visit the magnificent **Jerónimos Monastery (Mosteiro dos Jerónimos)**, a stunning example of Manueline architecture and the resting place of Vasco da Gama.
    *   Walk along the waterfront to see the iconic **Belém Tower (Torre de Belém)**, a fortress that once guarded the city's harbor, and the **Monument to the Discoveries (Padrão dos Descobrimentos)**, celebrating Portugal's explorers.
*   **Lunch (1:00 PM - 2:30 PM): Iconic Pastéis de Belém & Local Bites**
    *   Indulge in the world-famous **Pastéis de Belém** at the original factory, a must-try culinary experience in Lisbon.
    *   Enjoy a light and traditional Portuguese lunch at a nearby café or eatery in Belém, perhaps trying a "Bifana" (pork sandwich) or a "Caldo Verde" (kale soup).
*   **Afternoon (2:30 PM - 5:30 PM): Time Out Market & Trendy LX Factory**
    *   Travel back towards the city center to Cais do Sodré and immerse yourself in the vibrant atmosphere of the **Time Out Market (Mercado da Ribeira)**. This historic market now houses a large food hall with stalls from some of Lisbon's best chefs and restaurants, offering a fantastic opportunity to sample various local and gourmet foods.
    *   Alternatively, for a different vibe, you could visit **LX Factory**, an industrial area transformed into a trendy hub of unique shops, art studios, and restaurants, perfect for a stroll and a coffee.
*   **Evening (7:00 PM onwards): Dinner in Principe Real or Chiado & Sunset Views**
    *   Savor your final dinner in Lisbon in the elegant **Principe Real** or lively **Chiado** districts, known for their diverse culinary scene ranging from traditional taverns to modern bistros.
    *   Before or after dinner, consider catching a memorable sunset view from a different vantage point, such as **Miradouro de São Pedro de Alcântara**, offering panoramic views over Baixa and São Jorge Castle.

How does this plan for Day 2 sound for concluding your trip to Lisbon?

--------------------------------------------------



### Scenario 3b: Agent WITHOUT Memory (Using SEPARATE Sessions) ❌

Now, let's see what happens if we mess up our session management. Here, we'll give the agent a case of amnesia by creating a **brand new, separate session for each turn**.

Pay close attention to the agent's response to the second query. Because it's in a new session, it has no memory of the trip to Lisbon we just discussed!

In [12]:
# --- Scenario 2b: Demonstrating Memory FAILURE ---

async def run_memory_failure_demonstration():
    print("\n" + "#"*60)
    print("### 🧠 DEMO 2b: AGENT WITH AMNESIA (SEPARATE SESSIONS) ###")
    print("#"*60)

    # --- Turn 1: The user initiates the trip in the FIRST session ---
    query1 = "Hi! I want to plan a 2-day trip to Lisbon, Portugal. I'm interested in historic sites and great local food."
    session_one = await session_service.create_session(
        app_name=multi_day_agent.name,
        user_id=my_user_id
    )
    print(f"\nCreated a session for Turn 1: {session_one.id}")
    print(f"🗣️ User (Turn 1): '{query1}'")
    await run_agent_query(multi_day_agent, query1, session_one, my_user_id)

    # --- Turn 2: The user asks to continue... but in a completely NEW session ---
    query2 = "Yes, that looks perfect! Please plan Day 2."
    session_two = await session_service.create_session(
        app_name=multi_day_agent.name,
        user_id=my_user_id
    )
    print(f"\nCreated a BRAND NEW session for Turn 2: {session_two.id}")
    print(f"🗣️ User (Turn 2): '{query2}'")
    await run_agent_query(multi_day_agent, query2, session_two, my_user_id)

await run_memory_failure_demonstration()


############################################################
### 🧠 DEMO 2b: AGENT WITH AMNESIA (SEPARATE SESSIONS) ###
############################################################

Created a session for Turn 1: 5813e42b-9d4b-49ef-8a1c-4548a1a522ee
🗣️ User (Turn 1): 'Hi! I want to plan a 2-day trip to Lisbon, Portugal. I'm interested in historic sites and great local food.'

🚀 Running query for agent: 'multi_day_trip_agent' in session: '5813e42b-9d4b-49ef-8a1c-4548a1a522ee'...
EVENT: model_version='gemini-2.5-flash' content=Content(
  parts=[
    Part(
      text="""Hello! A fantastic choice! Lisbon is a beautiful city with so much to offer, especially if you're interested in history and local food.

Let's plan your first day. How about this for Day 1?

**Day 1: Alfama and Baixa's Historic Charms & Culinary Delights**

*   **Morning (9:00 AM - 1:00 PM): Explore Alfama District**
    *   Start your day by wandering through the labyrinthine streets of Alfama, Lisbon's oldest district.
  

Hello! A fantastic choice! Lisbon is a beautiful city with so much to offer, especially if you're interested in history and local food.

Let's plan your first day. How about this for Day 1?

**Day 1: Alfama and Baixa's Historic Charms & Culinary Delights**

*   **Morning (9:00 AM - 1:00 PM): Explore Alfama District**
    *   Start your day by wandering through the labyrinthine streets of Alfama, Lisbon's oldest district.
    *   Visit the **Lisbon Cathedral (Sé de Lisboa)**, a majestic Romanesque cathedral with a rich history.
    *   Walk up to the **Miradouro das Portas do Sol** and **Miradouro de Santa Luzia** for breathtaking views over Alfama and the Tagus River.
    *   Explore **São Jorge Castle (Castelo de São Jorge)**, a Moorish castle offering panoramic views and a glimpse into Lisbon's past.
*   **Lunch (1:00 PM - 2:30 PM): Traditional Portuguese Lunch in Alfama**
    *   Enjoy a traditional Portuguese lunch at a local tasca in Alfama, perhaps trying "Bacalhau à Brás" (codfish with scrambled eggs, potatoes, and onions) or grilled sardines.
*   **Afternoon (2:30 PM - 6:00 PM): Baixa District and Rossio Square**
    *   Head down to the Baixa district, rebuilt after the 1755 earthquake with a grid-like street plan.
    *   Stroll along **Rua Augusta** and admire the architecture, leading to the magnificent **Praça do Comércio** (Commerce Square) by the riverfront.
    *   Visit **Rossio Square (Praça de D. Pedro IV)**, a lively central square with beautiful fountains and historic buildings.
*   **Evening (7:30 PM onwards): Dinner & Fado in Alfama or Bairro Alto**
    *   Indulge in a delicious dinner, perhaps trying another local specialty.
    *   Consider experiencing a traditional Fado show with dinner, a soulful Portuguese musical genre, often found in Alfama or the Bairro Alto district.

How does this sound for your first day in Lisbon?

--------------------------------------------------


Created a BRAND NEW session for Turn 2: fb003c91-c6b3-44a3-89c7-8fe9b504219b
🗣️ User (Turn 2): 'Yes, that looks perfect! Please plan Day 2.'

🚀 Running query for agent: 'multi_day_trip_agent' in session: 'fb003c91-c6b3-44a3-89c7-8fe9b504219b'...
EVENT: model_version='gemini-2.5-flash' content=Content(
  parts=[
    Part(
      text="""Great! I'm glad Day 1 looks good. Let's move on to Day 2 in Paris, focusing on more artistic and historical gems.

### Day 2: Bohemian Charm & Gothic Splendor

*   **Morning (9:30 AM - 1:00 PM):** **Explore Montmartre.** Start your day by taking the funicular or walking up to the **Basilica of Sacré-Cœur** for breathtaking panoramic views of Paris. Afterwards, wander through the charming streets of Montmartre, soaking in the bohemian atmosphere at **Place du Tertre**, where artists display their works. Discover hidden staircases and quaint cafes.
*   **Lunch (1:00 PM - 2:00 PM):** Enjoy a traditional Fr

Great! I'm glad Day 1 looks good. Let's move on to Day 2 in Paris, focusing on more artistic and historical gems.

### Day 2: Bohemian Charm & Gothic Splendor

*   **Morning (9:30 AM - 1:00 PM):** **Explore Montmartre.** Start your day by taking the funicular or walking up to the **Basilica of Sacré-Cœur** for breathtaking panoramic views of Paris. Afterwards, wander through the charming streets of Montmartre, soaking in the bohemian atmosphere at **Place du Tertre**, where artists display their works. Discover hidden staircases and quaint cafes.
*   **Lunch (1:00 PM - 2:00 PM):** Enjoy a traditional French lunch at a bistro in Montmartre, perhaps trying a Croque Monsieur or a classic onion soup.
*   **Afternoon (2:30 PM - 5:30 PM):** **Discover Île de la Cité.** Head to the historical heart of Paris. Admire the exterior of the **Notre Dame Cathedral** (currently under reconstruction) and then visit the stunning **Sainte-Chapelle**, famous for its magnificent stained-glass windows.
*   **Evening (7:00 PM onwards):** **Marais District Dinner & Stroll.** Enjoy dinner in the trendy Marais district, known for its historic architecture, vibrant atmosphere, and diverse culinary scene. Afterwards, take a leisurely stroll through its charming streets.

How does this sound for your second day?

--------------------------------------------------



See? The agent was confused! It likely asked what destination or what trip we were talking about. Because the second query was in a fresh, isolated session, the agent had no memory of planning Day 1 in Lisbon.

This perfectly illustrates why **managing sessions is the key to building truly conversational agents!**

### 4.1 Multi-Agent Mayhem with `SequentialAgent` 🧠→⛓️→🤖

You've seen how to manually link agents together with custom Python code. It works, but it can get complicated. Now, let's refactor our workflow to use a powerful, built-in ADK feature designed specifically for this: the **`SequentialAgent`**.

The `SequentialAgent` is a *workflow agent*. It's not powered by an LLM itself; instead, its only job is to execute a list of other agents in a strict, predefined order.

The real magic ✨ is how it passes information. The ADK uses a shared `state` dictionary that each agent in the sequence can read from and write to.

**Our New Workflow:**

1.  **Foodie Agent**: Finds the restaurant and saves the name to `state['destination']`.
2.  **Transportation Agent**: Automatically reads `state['destination']` and uses it to find directions.

This means we no longer need custom Python code to extract text or build new queries! The ADK handles the plumbing for us.

```
+-------------------------------+
|  find_and_navigate_agent 🧭   |
| SequentialAgent:             |
| 1. Find destination          |
| 2. Get directions            |
+---------------+---------------+
                |
     +----------+----------+
     |                     |
     v                     v
+----------------+   +------------------------+
| foodie_agent 🍣 |   | transportation_agent 🚗 |
| Finds place     |   | Uses {destination}     |
| Output: 'Jin Sho'|   | Output: Directions     |
+----------------+   +------------------------+

Final Output: 🍣 Restaurant + 🚗 Route
```

In [17]:
from google.adk.agents import Agent, SequentialAgent, LoopAgent, ParallelAgent

In [18]:
# --- Agent Definitions for our Specialist Team (Refactored for Sequential Workflow) ---

# ✨ CHANGE 1: We tell foodie_agent to save its output to the shared state.
# Note the new `output_key` and the more specific instruction.
foodie_agent = Agent(
    name="foodie_agent",
    model="gemini-2.5-flash",
    tools=[google_search],
    instruction="""You are an expert food critic. Your goal is to find the best restaurant based on a user's request.

    When you recommend a place, you must output *only* the name of the establishment and nothing else.
    For example, if the best sushi is at 'Jin Sho', you should output only: Jin Sho
    """,
    output_key="destination"  # ADK will save the agent's final response to state['destination']
)

# ✨ CHANGE 2: We tell transportation_agent to read from the shared state.
# The `{destination}` placeholder is automatically filled by the ADK from the state.
transportation_agent = Agent(
    name="transportation_agent",
    model="gemini-2.5-flash",
    tools=[google_search],
    instruction="""You are a navigation assistant. Given a destination, provide clear directions.
    The user wants to go to: {destination}.

    Analyze the user's full original query to find their starting point.
    Then, provide clear directions from that starting point to {destination}.
    """,
)

# ✨ CHANGE 3: Define the SequentialAgent to manage the workflow.
# This agent will run foodie_agent, then transportation_agent, in that exact order.
find_and_navigate_agent = SequentialAgent(
    name="find_and_navigate_agent",
    sub_agents=[foodie_agent, transportation_agent],
    description="A workflow that first finds a location and then provides directions to it."
)

weekend_guide_agent = Agent(
    name="weekend_guide_agent",
    model="gemini-2.5-flash",
    tools=[google_search],
    instruction="You are a local events guide. Your task is to find interesting events, concerts, festivals, and activities happening on a specific weekend."
)

# --- The Brain of the Operation: The Router Agent ---

# We update the router to know about our new, powerful SequentialAgent.
router_agent = Agent(
    name="router_agent",
    model="gemini-2.5-flash",
    instruction="""
    You are a request router. Your job is to analyze a user's query and decide which of the following agents or workflows is best suited to handle it.
    Do not answer the query yourself, only return the name of the most appropriate choice.

    Available Options:
    - 'foodie_agent': For queries *only* about food, restaurants, or eating.
    - 'weekend_guide_agent': For queries about events, concerts, or activities happening on a specific timeframe like a weekend.
    - 'day_trip_agent': A general planner for any other day trip requests.
    - 'find_and_navigate_agent': Use this for complex queries that ask to *first find a place* and *then get directions* to it.

    Only return the single, most appropriate option's name and nothing else.
    """
)

# We create a dictionary of all our executable agents for easy lookup.
# This now includes our powerful new workflow agent!
worker_agents = {
    "day_trip_agent": day_trip_agent,
    "foodie_agent": foodie_agent,
    "weekend_guide_agent": weekend_guide_agent,
    "find_and_navigate_agent": find_and_navigate_agent, # Add the new sequential agent
}

print("🤖 Agent team assembled with a SequentialAgent workflow!")

🤖 Agent team assembled with a SequentialAgent workflow!


In [19]:
# --- Let's Test the Streamlined Workflow! ---

async def run_sequential_app():
    queries = [
        "I want to eat the best sushi in Palo Alto.", # Should go to foodie_agent
        "Are there any cool outdoor concerts this weekend?", # Should go to weekend_guide_agent
        "Find me the best sushi in Palo Alto and then tell me how to get there from the Caltrain station." # Should trigger the SequentialAgent
    ]

    for query in queries:
        print(f"\n{'='*60}\n🗣️ Processing New Query: '{query}'\n{'='*60}")

        # 1. Ask the Router Agent to choose the right agent or workflow
        router_session = await session_service.create_session(app_name=router_agent.name, user_id=my_user_id)
        print("🧠 Asking the router agent to make a decision...")
        chosen_route = await run_agent_query(router_agent, query, router_session, my_user_id, is_router=True)
        chosen_route = chosen_route.strip().replace("'", "")
        print(f"🚦 Router has selected route: '{chosen_route}'")

        # 2. Execute the chosen route
        # This logic is now much simpler! The SequentialAgent is treated just like any other worker.
        if chosen_route in worker_agents:
            worker_agent = worker_agents[chosen_route]
            print(f"--- Handing off to {worker_agent.name} ---")
            worker_session = await session_service.create_session(app_name=worker_agent.name, user_id=my_user_id)
            await run_agent_query(worker_agent, query, worker_session, my_user_id)
            print(f"--- {worker_agent.name} Complete ---")
        else:
            print(f"🚨 Error: Router chose an unknown route: '{chosen_route}'")

await run_sequential_app()


🗣️ Processing New Query: 'I want to eat the best sushi in Palo Alto.'
🧠 Asking the router agent to make a decision...

🚀 Running query for agent: 'router_agent' in session: 'f59468b4-58e1-4865-ba46-148eaf7f2867'...
🚦 Router has selected route: 'foodie_agent'
--- Handing off to foodie_agent ---

🚀 Running query for agent: 'foodie_agent' in session: 'f011173c-b35b-4f93-9bfb-f42e84d16b3c'...
EVENT: model_version='gemini-2.5-flash' content=Content(
  parts=[
    Part(
      text='Jin Sho'
    ),
  ],
  role='model'
) grounding_metadata=GroundingMetadata(
  search_entry_point=SearchEntryPoint(
    rendered_content="""<style>
.container {
  align-items: center;
  border-radius: 8px;
  display: flex;
  font-family: Google Sans, Roboto, sans-serif;
  font-size: 14px;
  line-height: 20px;
  padding: 8px 12px;
}
.chip {
  display: inline-block;
  border: solid 1px;
  border-radius: 16px;
  min-width: 14px;
  padding: 5px 16px;
  text-align: center;
  user-select: none;
  margin: 0 8px;
  -webki

Jin Sho

--------------------------------------------------

--- foodie_agent Complete ---

🗣️ Processing New Query: 'Are there any cool outdoor concerts this weekend?'
🧠 Asking the router agent to make a decision...

🚀 Running query for agent: 'router_agent' in session: '7433ad27-b5bf-4322-b450-d6e2648fa4de'...
🚦 Router has selected route: 'weekend_guide_agent'
--- Handing off to weekend_guide_agent ---

🚀 Running query for agent: 'weekend_guide_agent' in session: '2ec98929-5d38-4e3f-9dab-7876603492cd'...
EVENT: model_version='gemini-2.5-flash' content=Content(
  parts=[
    Part(
      text="""Yes, there are a few cool outdoor concerts happening this weekend, March 21st and 22nd, 2026, across different locations:

*   **League City, Texas:** On Saturday, March 21st, "A Day of Music" will be held at League Park. This free event will feature various local musicians and will be headlined by the Grammy-nominated duo "Trout Fishing in America," performing from 7:30 p.m. to 9 p.m..
*   **San Diego,

Yes, there are a few cool outdoor concerts happening this weekend, March 21st and 22nd, 2026, across different locations:

*   **League City, Texas:** On Saturday, March 21st, "A Day of Music" will be held at League Park. This free event will feature various local musicians and will be headlined by the Grammy-nominated duo "Trout Fishing in America," performing from 7:30 p.m. to 9 p.m..
*   **San Diego, California:** SeaWorld San Diego's Summer Concert Series features The Band Perry performing at the Bayside Amphitheater on Saturday, March 21st, at 6:00 p.m..
*   **St. Augustine, Florida:** Widespread Panic has sold-out concerts at the St. Augustine Amphitheatre on both Saturday, March 21st, and Sunday, March 22nd. Both shows are scheduled to start at 7:00 p.m. on Saturday and 6:30 p.m. on Sunday.
*   **Houston, Texas:**
    *   On Saturday, March 21st, Bone Thugs n Harmony will perform an outdoor concert at Home Run Dugout.
    *   Also on Saturday, March 21st, the 3rd Annual Girls on the Green will take place at Discovery Green, featuring the local all-female rock band Garbage Girlfriend.
    *   On Sunday, March 22nd, the alt-rock indie band Rainbow Kitten Surprise will be performing on the lawn of White Oak Music Hall.

To help me find events more tailored to you in the future, please let me know your preferred location!

--------------------------------------------------

--- weekend_guide_agent Complete ---

🗣️ Processing New Query: 'Find me the best sushi in Palo Alto and then tell me how to get there from the Caltrain station.'
🧠 Asking the router agent to make a decision...

🚀 Running query for agent: 'router_agent' in session: 'e2bae92a-e76c-48de-93f3-578335a2e90e'...
🚦 Router has selected route: 'find_and_navigate_agent'
--- Handing off to find_and_navigate_agent ---

🚀 Running query for agent: 'find_and_navigate_agent' in session: 'a93b51b3-58ac-418e-8615-ebada180738d'...
EVENT: model_version='gemini-2.5-flash' content=Content(
  parts=[
    Part(
      text="""Jin Sho

To get to Jin Sho from the Palo Alto Caltrain station, you have several options:

**Walking:**
Jin Sho is approximately 1.6 miles from the Caltrain station, and the walk takes about 31 minutes.
1. From the Palo Alto Caltrain Station (95 University Ave), head southeast on University Avenue.
2. Turn right onto Alma Street.
3. Turn 

One highly-rated sushi restaurant in Palo Alto is Jin Sho.

To get to Jin Sho from the Palo Alto Caltrain station, you have several options:

**Walking:**
Jin Sho is approximately 1.6 miles from the Caltrain station, and the walk takes about 31 minutes.
1. From the Palo Alto Caltrain Station (95 University Ave), head southeast on University Avenue.
2. Turn right onto Alma Street.
3. Turn left onto California Avenue.
4. Jin Sho will be on your left at 454 California Avenue.

**Bus:**
You can take the Line 21 bus from the Palo Alto Transit Center (near the Caltrain station) to Middlefield & Embarcadero. This journey takes approximately 10 minutes and costs between $2 and $5.

**Taxi/Rideshare:**
A taxi or rideshare service will take approximately 3 minutes and cost around $12-$15.

--------------------------------------------------

--- find_and_navigate_agent Complete ---


### Running the Streamlined App

Notice how much simpler the code below is. There is no longer a special `if chosen_route == 'find_and_navigate_combo':` block with custom logic.

The `find_and_navigate_agent` is now a self-contained unit. We can treat it just like any other agent, hand it the query, and trust the `SequentialAgent` to handle all the internal steps. This makes our main application code much cleaner and easier to read.

---
## 4.2 Iterative Ideas with `LoopAgent` 🧠→🔁→🤖

Sometimes a task isn't a straight line; it's a loop of refinement. A user might ask for a plan, but have constraints that require checking and re-planning. For this, the ADK provides the **`LoopAgent`**.

The `LoopAgent` executes a sequence of sub-agents repeatedly until a condition is met. This is perfect for workflows involving trial and error, like planning a trip with a tight schedule.

**Our New Workflow: The Perfectionist Planner**

1. **Planner Agent**: Proposes an itinerary (e.g., a museum and a restaurant).
2. **Critic Agent**: Checks the plan against a constraint (e.g., "Is the travel time between these two places less than 30 minutes?").
3. **Refiner Agent**: If the critic finds a problem, this agent takes the feedback and creates a new, improved plan. If the critic is happy, it calls a special `exit_loop` tool to stop the process.

The `LoopAgent` manages this cycle, ensuring we don't get stuck in an infinite loop by setting a `max_iterations` limit.

```
+-------------------------------+
|  iterative_planner_agent 🔁   |
| SequentialAgent:              |
| 1. Propose Plan               |
| 2. Refine in loop (≤ 3 times) |
+---------------+---------------+
                |
     +----------+----------+
     |                     |
     v                     v
+----------------+   +-----------------------+
| planner_agent  |   | refinement_loop 🔁     |
| Propose plan   |   | LoopAgent             |
| e.g., Activity +  | 1. Critic (time check) |
| Restaurant       | 2. Refiner (fix/exit)   |
+----------------+   +-----------------------+

Uses shared state: {current_plan}, {criticism}
Exits when: "Plan is feasible..."

```

In [20]:
# --- Agent Definitions for an Iterative Workflow ---

# A tool to signal that the loop should terminate
COMPLETION_PHRASE = "The plan is feasible and meets all constraints."
def exit_loop(tool_context: ToolContext):
  """Call this function ONLY when the plan is approved, signaling the loop should end."""
  print(f"  [Tool Call] exit_loop triggered by {tool_context.agent_name}")
  tool_context.actions.escalate = True
  return {}

# Agent 1: Proposes an initial plan
planner_agent = Agent(
    name="planner_agent", model="gemini-2.5-flash", tools=[google_search],
    instruction="You are a trip planner. Based on the user's request, propose a single activity and a single restaurant. Output only the names, like: 'Activity: Exploratorium, Restaurant: La Mar'.",
    output_key="current_plan"
)

# Agent 2 (in loop): Critiques the plan
critic_agent = Agent(
    name="critic_agent", model="gemini-2.5-flash", tools=[google_search],
    instruction=f"""You are a logistics expert. Your job is to critique a travel plan. The user has a strict constraint: total travel time must be short.
    Current Plan: {{current_plan}}
    Use your tools to check the travel time between the two locations.
    IF the travel time is over 45 minutes, provide a critique, like: 'This plan is inefficient. Find a restaurant closer to the activity.'
    ELSE, respond with the exact phrase: '{COMPLETION_PHRASE}'""",
    output_key="criticism"
)

# Agent 3 (in loop): Refines the plan or exits
refiner_agent = Agent(
    name="refiner_agent", model="gemini-2.5-flash", tools=[google_search, exit_loop],
    instruction=f"""You are a trip planner, refining a plan based on criticism.
    Original Request: {{session.query}}
    Critique: {{criticism}}
    IF the critique is '{COMPLETION_PHRASE}', you MUST call the 'exit_loop' tool.
    ELSE, generate a NEW plan that addresses the critique. Output only the new plan names, like: 'Activity: de Young Museum, Restaurant: Nopa'.""",
    output_key="current_plan"
)

# ✨ The LoopAgent orchestrates the critique-refine cycle ✨
refinement_loop = LoopAgent(
    name="refinement_loop",
    sub_agents=[critic_agent, refiner_agent],
    max_iterations=3
)

# ✨ The SequentialAgent puts it all together ✨
iterative_planner_agent = SequentialAgent(
    name="iterative_planner_agent",
    sub_agents=[planner_agent, refinement_loop],
    description="A workflow that iteratively plans and refines a trip to meet constraints."
)

print("🤖 Agent team updated with an iterative LoopAgent workflow!")

🤖 Agent team updated with an iterative LoopAgent workflow!


---
## Parallel Power with `ParallelAgent` 🧠→⚡️→🤖🤖🤖

What if a user wants to find multiple, unrelated things at once? "Find me a museum, a concert, AND a restaurant." Running these searches one by one is slow and inefficient.

Enter the **`ParallelAgent`**. This workflow agent executes a list of sub-agents *concurrently*, dramatically speeding up tasks that can be performed independently.

**Our New Workflow: The Multi-Researcher**

1.  **Parallel Agent**: Simultaneously runs three specialist agents:
    - `MuseumFinderAgent`: Finds a museum.
    - `ConcertFinderAgent`: Finds a concert.
    - `FoodieAgent`: Finds a restaurant.
2.  **Synthesis Agent**: Once all three parallel searches are complete, this final agent gathers the results (which were saved to the shared `state`) and formats them into a single, neat summary for the user.

This pattern lets us get a lot of work done, fast! 🚀

```
+-------------------------------+
|  parallel_planner_agent ⚡     |
| SequentialAgent:              |
| 1. Run parallel research      |
| 2. Synthesize results         |
+---------------+---------------+
                |
     +----------+----------------------+
     |                                 |
     v                                 v
+-------------------------+       +-----------------------------+
| parallel_research_agent ⚡   |   | synthesis_agent 📋          |
| ParallelAgent:              |   | Combine results            |
| - museum_finder_agent 🖼️     |   | Output: Bulleted summary   |
| - concert_finder_agent 🎵    |   +-----------------------------+
| - restaurant_finder_agent 🍽️ |
+-------------------------+

Final Output:
• Museum: XYZ  
• Concert: Artist at Venue  
• Restaurant: ABC
```

In [21]:
# --- Agent Definitions for a Parallel Workflow ---

# Specialist Agent 1
museum_finder_agent = Agent(
    name="museum_finder_agent", model="gemini-2.5-flash", tools=[google_search],
    instruction="You are a museum expert. Find the best museum based on the user's query. Output only the museum's name.",
    output_key="museum_result"
)

# Specialist Agent 2
concert_finder_agent = Agent(
    name="concert_finder_agent", model="gemini-2.5-flash", tools=[google_search],
    instruction="You are an events guide. Find a concert based on the user's query. Output only the concert name and artist.",
    output_key="concert_result"
)

# We can reuse our foodie_agent for the third parallel task!
# Just need to give it a new output_key for this workflow.
# restaurant_finder_agent = foodie_agent.copy(update={"output_key": "restaurant_result"})
restaurant_finder_agent = Agent(
    name="restaurant_finder_agent",
    model="gemini-2.5-flash",
    tools=[google_search],
    instruction="""You are an expert food critic. Your goal is to find the best restaurant based on a user's request.

    When you recommend a place, you must output *only* the name of the establishment.
    For example, if the best sushi is at 'Jin Sho', you should output only: Jin Sho
    """,
    output_key="restaurant_result" # Set the correct output key for this workflow
)


# ✨ The ParallelAgent runs all three specialists at once ✨
parallel_research_agent = ParallelAgent(
    name="parallel_research_agent",
    sub_agents=[museum_finder_agent, concert_finder_agent, restaurant_finder_agent]
)

# Agent to synthesize the parallel results
synthesis_agent = Agent(
    name="synthesis_agent", model="gemini-2.5-flash",
    instruction="""You are a helpful assistant. Combine the following research results into a clear, bulleted list for the user.
    - Museum: {museum_result}
    - Concert: {concert_result}
    - Restaurant: {restaurant_result}
    """
)

# ✨ The SequentialAgent runs the parallel search, then the synthesis ✨
parallel_planner_agent = SequentialAgent(
    name="parallel_planner_agent",
    sub_agents=[parallel_research_agent, synthesis_agent],
    description="A workflow that finds multiple things in parallel and then summarizes the results."
)

print("🤖 Agent team supercharged with a ParallelAgent workflow!")

🤖 Agent team supercharged with a ParallelAgent workflow!


---
### 4.4 Final Step: Updating the Router and Running the App

Now we just have one last thing to do: make our `router_agent` aware of these powerful new workflows! We'll add `iterative_planner_agent` and `parallel_planner_agent` to its list of available options.

Then we can run our app with new queries designed to trigger these advanced, multi-agent workflows.

```
                    +---------------------+
                    |    User Query 🗣️     |
                    +----------+----------+
                               |
                               v
                    +---------------------+
                    |   Router Agent 🤖    |
                    |  (Classify Request) |
                    +----------+----------+
                               |
      +-----------+-----------+-----------+-----------+------------+
      |           |           |           |           |            |
      v           v           v           v           v            v
+-------------+  +------------------+  +------------------+  +------------------+  +-----------------+
| foodie_agent|  | find_and_navigate|  | iterative_planner|  | parallel_planner |  | day_trip_agent  |
| 🍣 Food Only |  | 🧭 Seq Workflow   |  | 🔁 Loop Workflow  |  | ⚡ Parallel Tasks |  | 🧳 Basic Plan     |
+-------------+  +------------------+  +------------------+  +------------------+  +-----------------+
```

In [22]:
# --- The ULTIMATE Router Agent --- #

router_agent = Agent(
    name="router_agent",
    model="gemini-2.5-flash",
    instruction="""
    You are a master request router. Your job is to analyze a user's query and decide which of the following agents or workflows is best suited to handle it.
    Do not answer the query yourself, only return the name of the most appropriate choice.

    Available Options:
    - 'foodie_agent': For queries *only* about finding a single food place.
    - 'find_and_navigate_agent': For queries that ask to *first find a place* and *then get directions* to it.
    - 'iterative_planner_agent': For planning a trip with a specific constraint that needs checking, like travel time.
    - 'parallel_planner_agent': For queries that ask to find multiple, independent things at once (e.g., a museum AND a concert AND a restaurant).
    - 'day_trip_agent': A general planner for any other simple day trip requests.

    Only return the single, most appropriate option's name and nothing else.
    """
)

# The master dictionary of all our executable agents and workflows
worker_agents = {
    "day_trip_agent": day_trip_agent,
    "foodie_agent": foodie_agent, # For simple food queries
    "find_and_navigate_agent": find_and_navigate_agent, # Sequential
    "iterative_planner_agent": iterative_planner_agent, # Loop
    "parallel_planner_agent": parallel_planner_agent,   # Parallel
}

# --- Let's Test Everything! ---

async def run_fully_loaded_app():
    queries = [
        # Test Case 1: Simple Sequential Flow
        "Find me the best sushi in Palo Alto and then tell me how to get there from the Caltrain station.",

        # Test Case 2: Iterative Loop Flow
        "Plan me a day in San Francisco with a museum and a nice dinner, but make sure the travel time between them is very short.",

        # Test Case 3: Parallel Flow
        "Help me plan a trip to SF. I need one museum, one concert, and one great restaurant."
    ]

    for query in queries:
        print(f"\n{'='*60}\n🗣️ Processing New Query: '{query}'\n{'='*60}")

        # 1. Ask the Router Agent to choose the right agent or workflow
        router_session = await session_service.create_session(app_name=router_agent.name, user_id=my_user_id)
        print("🧠 Asking the router agent to make a decision...")
        chosen_route = await run_agent_query(router_agent, query, router_session, my_user_id, is_router=True)
        chosen_route = chosen_route.strip().replace("'", "")
        print(f"🚦 Router has selected route: '{chosen_route}'")

        # 2. Execute the chosen route
        if chosen_route in worker_agents:
            worker_agent = worker_agents[chosen_route]
            print(f"--- Handing off to {worker_agent.name} ---")
            worker_session = await session_service.create_session(app_name=worker_agent.name, user_id=my_user_id)
            await run_agent_query(worker_agent, query, worker_session, my_user_id)
            print(f"--- {worker_agent.name} Complete ---")
        else:
            print(f"🚨 Error: Router chose an unknown route: '{chosen_route}'")

await run_fully_loaded_app()


🗣️ Processing New Query: 'Find me the best sushi in Palo Alto and then tell me how to get there from the Caltrain station.'
🧠 Asking the router agent to make a decision...

🚀 Running query for agent: 'router_agent' in session: 'ecd73da6-a410-4f0c-9f18-6becf29cd561'...
🚦 Router has selected route: 'find_and_navigate_agent'
--- Handing off to find_and_navigate_agent ---

🚀 Running query for agent: 'find_and_navigate_agent' in session: 'bf8323cb-c2fa-40ae-8ca5-7bf684139062'...
EVENT: model_version='gemini-2.5-flash' content=Content(
  parts=[
    Part(
      text="""Fuki Sushi

To get to Fuki Sushi from the Palo Alto Caltrain Station, you will need to head south along El Camino Real.

Here are the directions:
1. Exit the Palo Alto Caltrain Station at 95 University Ave, Palo Alto, CA 94301.
2. Walk southwest on University Ave toward Alma St.
3. Turn left onto El Camino Real.
4. Continue on El Camino Real for approximately 2.5 miles.
5. Fuki Sushi will be on your right at 4119 El Camino Re

According to the foodie agent, Fuki Sushi is a recommended sushi restaurant in Palo Alto.

To get to Fuki Sushi from the Palo Alto Caltrain Station, you will need to head south along El Camino Real.

Here are the directions:
1.  Exit the Palo Alto Caltrain Station at 95 University Ave, Palo Alto, CA 94301.
2.  Walk southwest on University Ave toward Alma St.
3.  Turn left onto El Camino Real.
4.  Continue on El Camino Real for approximately 2.5 miles.
5.  Fuki Sushi will be on your right at 4119 El Camino Real, Palo Alto, CA 94306.

Please note that this distance is generally considered too far to walk for most people (around a 45-50 minute walk). A taxi, rideshare service, or a local bus (such as VTA) would be recommended.

--------------------------------------------------

--- find_and_navigate_agent Complete ---

🗣️ Processing New Query: 'Plan me a day in San Francisco with a museum and a nice dinner, but make sure the travel time between them is very short.'
🧠 Asking the router agent to make a decision...

🚀 Running query for agent: 'router_agent' in session: 'f7d21814-20df-44d4-b88a-5f12c1705b43'...
🚦 Router has selected route: 'iterative_planner_agent'
--- Handing off to iterative_planner_agent ---

🚀 Running query for agent: 'iterative_planner_agent' in session: '0e276ef4-70b9-4f9f-9595-a4d448ef0d77'...
EVENT: model_version='gemini-2.5-flash' content=Content(
  parts=[
    Part(
      text="""Activity: San Francisco Museum of Modern Art
Restaurant: One Market Restaurant"""
    ),
  ],
  role='model'
) grounding_metadata=GroundingMetadata(
  search_entry_point=SearchEntryPoint(
    rendered_content="""<style>
.container {
  align-items: center;
  border-radius: 8px;
  display: flex;
  font-family: G

EVENT: model_version='gemini-2.5-flash' content=Content(
  parts=[
    Part(
      text="""The San Francisco Museum of Modern Art and One Market Restaurant are both located in downtown San Francisco. The travel time between these two locations is short, well within the specified constraint.

*   **Walking:** The estimated walking time between SFMOMA and One Market Restaurant is approximately 37 minutes.
*   **Public Transportation (Muni/BART):** Taking public transportation, such as Muni Metro or BART, involves a short walk to a station (e.g., Montgomery Street Station from SFMOMA, around 5-7 minutes), a brief one-stop ride to Embarcadero Station (approximately 3-5 minutes), and then another short walk to One Market Restaurant (around 2-4 minutes). This totals roughly 10-16 minutes.
*   **Taxi/Rideshare:** A taxi or rideshare service would take approximately 3 minutes.

Given these travel times, none of which exceed 45 minutes, the plan is feasible and meets all constraints."""
    ),


An error occurred: 400 INVALID_ARGUMENT. {'error': {'code': 400, 'message': 'Built-in tools ({google_search}) and Function Calling cannot be combined in the same request. Please choose one to continue.', 'status': 'INVALID_ARGUMENT'}}

--------------------------------------------------

--- iterative_planner_agent Complete ---

🗣️ Processing New Query: 'Help me plan a trip to SF. I need one museum, one concert, and one great restaurant.'
🧠 Asking the router agent to make a decision...

🚀 Running query for agent: 'router_agent' in session: 'd0f4e441-337b-472a-bdf5-92e8cbdd886a'...
🚦 Router has selected route: 'parallel_planner_agent'
--- Handing off to parallel_planner_agent ---

🚀 Running query for agent: 'parallel_planner_agent' in session: 'eeab02de-d24b-4d1a-b958-5e5f84b509f8'...
EVENT: model_version='gemini-2.5-flash' content=Content(
  parts=[
    Part(
      text='Dogma with Frayle'
    ),
  ],
  role='model'
) grounding_metadata=GroundingMetadata(
  search_entry_point=SearchEntryPoint(
    rendered_content="""<style>
.container {
  align-items: center;
  border-radius: 8px;
  display: flex;
  font-family: Google Sans, Roboto, sans-serif;
  font-size: 14px;
  line-height: 20px;
  padding: 8px 12px;
}
.chip {
  

Here is a plan for your trip to San Francisco:

*   **Museum:** San Francisco Museum of Modern Art (SFMOMA)
*   **Concert:** Dogma with Frayle
*   **Restaurant:** State Bird Provisions

--------------------------------------------------

--- parallel_planner_agent Complete ---
